# Task 04 – Graph Neural Network Development

Two required architectures: GCN and GraphSAGE.

In [1]:
!pip install -q torch-geometric ogb streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 65.2 MB/s eta 0:00:00


In [3]:
# Libraries required for Task 04
import torch
import torch.nn.functional as F

from torch.nn import BatchNorm1d
from torch_geometric.nn import GCNConv, SAGEConv

from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/OGBN_Arxiv_Project')
RESULTS_ROOT = PROJECT_ROOT / 'results'

OUTPUT = RESULTS_ROOT / "task04_gnn_development"
OUTPUT.mkdir(parents=True, exist_ok=True)

In [5]:
import os
# Compatibility fix for trusted PyG objects downloaded by the official OGB package on PyTorch 2.6+.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

from google.colab import drive

# Check if drive is already mounted to avoid error on re-execution
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/OGBN_Arxiv_Project')
RESULTS_ROOT = PROJECT_ROOT / 'results'
MODELS_DIR = PROJECT_ROOT / 'models'
SRC_DIR = PROJECT_ROOT / 'src'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'

for folder in [PROJECT_ROOT, RESULTS_ROOT, MODELS_DIR, SRC_DIR, ARTIFACTS_DIR,
               PROJECT_ROOT / 'notebooks', PROJECT_ROOT / 'visualizations',
               PROJECT_ROOT / 'dashboard', PROJECT_ROOT / 'report',
               PROJECT_ROOT / 'presentation', PROJECT_ROOT / 'video']:
    folder.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)

Project root: /content/drive/MyDrive/OGBN_Arxiv_Project


In [7]:
# Common reproducibility settings
import warnings
import random
import numpy as np

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 4.1 GCN and GraphSAGE Models

The notebook implements the two architectures required by the lecturer. Both are simple two-layer models so their design and forward pass are easy to explain.

In [8]:
class GCN(torch.nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        dropout=0.5
    ):
        super().__init__()

        self.conv1 = GCNConv(
            input_dim,
            hidden_dim,
            cached=True
        )

        self.conv2 = GCNConv(
            hidden_dim,
            hidden_dim,
            cached=True
        )

        self.conv3 = GCNConv(
            hidden_dim,
            output_dim,
            cached=True
        )

        self.bn1 = BatchNorm1d(hidden_dim)
        self.bn2 = BatchNorm1d(hidden_dim)
        self.dropout = dropout

    def forward(
        self,
        features,
        edges,
        return_embeddings=False
    ):
        # First graph layer
        hidden1 = self.conv1(features, edges)
        hidden1 = self.bn1(hidden1)
        hidden1 = F.relu(hidden1)

        hidden1 = F.dropout(
            hidden1,
            p=self.dropout,
            training=self.training
        )

        # Second graph layer
        hidden2 = self.conv2(hidden1, edges)
        hidden2 = self.bn2(hidden2)
        hidden2 = F.relu(hidden2)

        # Residual connection
        hidden2 = hidden2 + hidden1

        hidden2 = F.dropout(
            hidden2,
            p=self.dropout,
            training=self.training
        )

        # Classification layer
        output = self.conv3(hidden2, edges)

        if return_embeddings:
            return output, hidden2

        return output


class GraphSAGE(torch.nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        dropout=0.5
    ):
        super().__init__()

        self.conv1 = SAGEConv(
            input_dim,
            hidden_dim,
            aggr="mean"
        )

        self.conv2 = SAGEConv(
            hidden_dim,
            hidden_dim,
            aggr="mean"
        )

        self.conv3 = SAGEConv(
            hidden_dim,
            output_dim,
            aggr="mean"
        )

        self.bn1 = BatchNorm1d(hidden_dim)
        self.bn2 = BatchNorm1d(hidden_dim)
        self.dropout = dropout

    def forward(
        self,
        features,
        edges,
        return_embeddings=False
    ):
        # First GraphSAGE layer
        hidden1 = self.conv1(features, edges)
        hidden1 = self.bn1(hidden1)
        hidden1 = F.relu(hidden1)

        hidden1 = F.dropout(
            hidden1,
            p=self.dropout,
            training=self.training
        )

        # Second GraphSAGE layer
        hidden2 = self.conv2(hidden1, edges)
        hidden2 = self.bn2(hidden2)
        hidden2 = F.relu(hidden2)

        # Residual connection
        hidden2 = hidden2 + hidden1

        hidden2 = F.dropout(
            hidden2,
            p=self.dropout,
            training=self.training
        )

        # Classification layer
        output = self.conv3(hidden2, edges)

        if return_embeddings:
            return output, hidden2

        return output


print("Three-layer GCN and GraphSAGE models defined.")

Three-layer GCN and GraphSAGE models defined.


## 4.2 Architecture Design

Each model uses three graph layers. The first layer produces hidden node representations, ReLU adds non-linearity, and dropout reduces overfitting. The second layer produces 40 class scores. GCN uses normalized graph convolution, while GraphSAGE uses mean neighbourhood aggregation.

## 4.3 Forward-Pass Verification

Instantiates both architectures and checks that each model produces one class-score vector per node with the expected output dimensions.

In [13]:
F_DIM = 16  # Example feature dimension
NUM_CLASSES = 2 # Example number of classes

toy_features = torch.randn(6, F_DIM)

toy_edges = torch.tensor([
    [0, 1, 2, 3, 4, 5],
    [1, 2, 3, 4, 5, 0]
])

for model_class in [GCN, GraphSAGE]:
    test_model = model_class(
        input_dim=F_DIM,
        hidden_dim=32,
        output_dim=NUM_CLASSES
    )

    test_output = test_model(
        toy_features,
        toy_edges
    )

    print(
        model_class.__name__,
        "output shape:",
        test_output.shape
    )

GCN output shape: torch.Size([6, 2])
GraphSAGE output shape: torch.Size([6, 2])
